<a id="logistic-model"></a>

$$
\text{逻辑回归 (Logistic Regression): 二分类模型, 输出 "属于类别 1" 的概率} \\ \quad \\
\text{线性回归的输出 } w^T x \in (-\infty, \infty) \text{, 概率必须在 } (0, 1) \text{ 内} \\
\text{思路: 用线性模型去拟合对数几率 (log-odds), 再用 sigmoid 反解出概率} \\
\ln \frac{p}{1 - p} = w^T x \implies p = P(y = 1 \mid x) = \sigma(w^T x) = \frac{1}{1 + e^{-w^T x}} \\
\text{(和线性回归一样, x 的第一个分量补 1, 偏置 b 并入 w)} \\ \quad \\
\text{分类规则: } p \geq 0.5 \iff w^T x \geq 0 \text{ 时预测为 1}
$$

数学基础: [Sigmoid 与对数几率](sigmoid_function.ipynb#log-odds) · [求导法则](../math_basics/calculus.ipynb#derivative-rules)

<a id="cross-entropy"></a>

$$
\text{损失函数: 由最大似然估计推出交叉熵 (Cross-Entropy)} \\ \quad \\
\text{每个样本的标签 } y_i \in \{0, 1\} \text{ 是一次伯努利试验, 成功概率 } p_i = \sigma(w^T x_i) \text{:} \\
P(y_i \mid x_i) = p_i^{y_i} (1 - p_i)^{1 - y_i} \quad (y_i = 1 \text{ 时为 } p_i \text{, } y_i = 0 \text{ 时为 } 1 - p_i) \\ \quad \\
\text{样本独立, 似然函数是乘积, 取对数变成求和:} \\
\ln L(w) = \sum_{i=1}^n \left[ y_i \ln p_i + (1 - y_i) \ln (1 - p_i) \right] \\ \quad \\
\text{最大化对数似然} \iff \text{最小化平均负对数似然 = 二元交叉熵 (Binary Cross-Entropy, BCE):} \\
J(w) = -\frac{1}{n} \sum_{i=1}^n \left[ y_i \ln p_i + (1 - y_i) \ln (1 - p_i) \right] \\ \quad \\
\text{为什么不用 MSE: } (y - \sigma(w^T x))^2 \text{ 关于 w 不是凸函数, 而且 } p \to 0 \text{ 或 } 1 \text{ 时梯度里有 } \sigma' \to 0 \text{, 学习很慢}
$$

数学基础: [伯努利分布的最大似然估计](../math_basics/statistics.ipynb#mle-bernoulli) · [最大似然估计](../math_basics/statistics.ipynb#mle)

<a id="logistic-gradient"></a>

$$
\text{梯度推导: 对单个样本用链式法则, 拆成三步} \\
\ell = -\left[ y \ln p + (1 - y) \ln(1 - p) \right], \quad p = \sigma(z), \quad z = w^T x \\ \quad \\
1. \ \frac{\partial \ell}{\partial p} = -\frac{y}{p} + \frac{1 - y}{1 - p} = \frac{-y(1 - p) + (1 - y) p}{p (1 - p)} = \frac{p - y}{p (1 - p)} \\
2. \ \frac{\partial p}{\partial z} = \sigma'(z) = \sigma(z)(1 - \sigma(z)) = p(1 - p) \\
3. \ \frac{\partial z}{\partial w} = x \\ \quad \\
\frac{\partial \ell}{\partial w} = \frac{\partial \ell}{\partial p} \cdot \frac{\partial p}{\partial z} \cdot \frac{\partial z}{\partial w} = \frac{p - y}{p (1 - p)} \cdot p (1 - p) \cdot x = (p - y) \, x \\
\text{(} p(1 - p) \text{ 正好约掉: 这就是交叉熵配 sigmoid 的好处, 梯度不会因为 } \sigma' \to 0 \text{ 而消失)} \\ \quad \\
\text{全部样本, 写成矩阵形式:} \\
\nabla J(w) = \frac{1}{n} \sum_{i=1}^n (p_i - y_i) x_i = \frac{1}{n} X^T (\sigma(X w) - y) \\
\text{和线性回归 MSE 的梯度 } \frac{2}{n} X^T (X w - y) \text{ 形式完全一样: "预测 - 真实" 乘以输入}
$$

数学基础: [链式法则](../math_basics/calculus.ipynb#derivative-rules) · [Sigmoid 的导数](sigmoid_function.ipynb) · 对比: [线性回归的梯度](linear_regression.ipynb#normal-equation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# 两个有重叠的二维正态类别 (完全可分的数据没有有限的最优解, w 会趋于无穷)
rng = np.random.default_rng(0)
n_half = 150
X0 = rng.multivariate_normal([-1.0, -0.5], [[1.0, 0.3], [0.3, 1.0]], n_half)
X1 = rng.multivariate_normal([1.2, 1.0], [[1.0, -0.2], [-0.2, 1.0]], n_half)
X_raw = np.vstack([X0, X1])
y = np.r_[np.zeros(n_half), np.ones(n_half)]
X = np.column_stack([np.ones(len(y)), X_raw])     # 第一列是 1

def bce(w, X, y):
    p = sigmoid(X @ w)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def bce_grad(w, X, y):
    return X.T @ (sigmoid(X @ w) - y) / len(y)

w_test = np.array([0.2, -0.5, 0.8])
h = 1e-6
num_grad = np.array([(bce(w_test + e, X, y) - bce(w_test - e, X, y)) / (2 * h) for e in np.eye(3) * h])
print(f'梯度 推导 = {bce_grad(w_test, X, y)}')
print(f'梯度 数值 = {num_grad}')

# 单个样本: 三步链式法则的每一项
x_i, y_i = X[0], y[0]
z_i = w_test @ x_i
p_i = sigmoid(z_i)
dl_dp = (p_i - y_i) / (p_i * (1 - p_i))
dp_dz = p_i * (1 - p_i)
print(f'单个样本: dl/dp * dp/dz = {dl_dp * dp_dz:.6f}, p - y = {p_i - y_i:.6f}')

<a id="logistic-hessian"></a>

$$
\text{Hessian 与凸性 (Convexity)} \\
\text{对梯度 } \frac{1}{n} \sum_i (p_i - y_i) x_i \text{ 再求一次导, y 是常数, } \frac{\partial p_i}{\partial w} = p_i (1 - p_i) x_i \text{:} \\
H = \nabla^2 J = \frac{1}{n} \sum_{i=1}^n p_i (1 - p_i) \, x_i x_i^T = \frac{1}{n} X^T S X, \quad S = \text{diag}(p_1 (1 - p_1), \cdots, p_n (1 - p_n)) \\ \quad \\
\text{半正定: 对任意 v, } v^T X^T S X v = \sum_i \underbrace{p_i (1 - p_i)}_{> 0} (x_i^T v)^2 \geq 0 \\
\text{所以 J 是凸函数, 没有局部最优陷阱, 驻点就是全局最小} \\ \quad \\
\text{梯度下降的学习率: 因为 } p(1 - p) \leq \frac{1}{4} \text{ (在 p = 0.5 时取最大)} \\
v^T H v \leq \frac{1}{4n} v^T X^T X v \implies \lambda_{\max}(H) \leq \frac{\lambda_{\max}(X^T X)}{4n} \\
\text{取 } \eta = \frac{4n}{\lambda_{\max}(X^T X)} \text{, 一定满足 } \eta \leq \frac{1}{\lambda_{\max}(H)} < \frac{2}{\lambda_{\max}(H)}
$$

数学基础: [正定矩阵](../math_basics/linear_algebra.ipynb#positive-definite) · [极值的二阶条件](../math_basics/calculus.ipynb#extremum) · [梯度下降的学习率上限](../math_basics/calculus.ipynb#gradient-descent)

In [ ]:
def bce_hessian(w, X):
    p = sigmoid(X @ w)
    return X.T @ (X * (p * (1 - p))[:, None]) / len(X)     # X^T S X / n, 不显式构造 n x n 的对角矩阵

H_test = bce_hessian(w_test, X)
num_H = np.array([(bce_grad(w_test + e, X, y) - bce_grad(w_test - e, X, y)) / (2 * h) for e in np.eye(3) * h]).T
print(f'Hessian 推导 == 数值? {np.allclose(H_test, num_H, atol=1e-6)}, 特征值 = {np.linalg.eigvalsh(H_test)} (全部 >= 0)')

lam_H_bound = np.linalg.eigvalsh(X.T @ X).max() / (4 * len(y))
print(f'lambda_max(H) = {np.linalg.eigvalsh(H_test).max():.4f} <= lambda_max(X^T X) / 4n = {lam_H_bound:.4f}')

# 凸性: 沿任意两点连线, J 在弦的下方
w_a, w_b = np.array([-2.0, 3.0, -1.0]), np.array([1.0, 0.5, 2.0])
ts = np.linspace(0, 1, 11)
on_line = np.array([bce((1 - t) * w_a + t * w_b, X, y) for t in ts])
chord = (1 - ts) * bce(w_a, X, y) + ts * bce(w_b, X, y)
print(f'J((1-t)a + t b) <= (1-t) J(a) + t J(b) 对所有 t 成立? {np.all(on_line <= chord + 1e-12)}')

<a id="newton-irls"></a>

$$
\text{求解: 梯度下降 vs 牛顿法 (IRLS)} \\ \quad \\
\text{1. 梯度下降: } w \leftarrow w - \eta \frac{1}{n} X^T (p - y) \\ \quad \\
\text{2. 牛顿法: } w \leftarrow w - H^{-1} \nabla J = w + (X^T S X)^{-1} X^T (y - p) \quad (\text{两个 } \frac{1}{n} \text{ 约掉}) \\ \quad \\
\text{改写成加权最小二乘 (Iteratively Reweighted Least Squares, IRLS):} \\
w_{new} = (X^T S X)^{-1} \left[ X^T S X w + X^T (y - p) \right] = (X^T S X)^{-1} X^T S \underbrace{\left[ X w + S^{-1} (y - p) \right]}_{\tilde{z} \text{ (工作响应 working response)}} \\
\text{对比加权最小二乘 } \min \sum_i s_i (\tilde{z}_i - x_i^T w)^2 \text{ 的解 } (X^T S X)^{-1} X^T S \tilde{z} \text{: 完全一样} \\
\text{每一步都是一个线性回归, 只是权重 S 和响应 } \tilde{z} \text{ 随 w 更新, 所以叫 "迭代重加权"} \\ \quad \\
\text{牛顿法二次收敛, 通常几步就到最优; 梯度下降每步便宜但需要很多步}
$$

数学基础: [牛顿法](../math_basics/calculus.ipynb#newton) · [多元泰勒展开](../math_basics/calculus.ipynb#multivariate-taylor) · [正规方程](linear_regression.ipynb#normal-equation)

In [ ]:
# 1. 梯度下降
eta = 4 * len(y) / np.linalg.eigvalsh(X.T @ X).max()
w_gd = np.zeros(3)
loss_gd = [bce(w_gd, X, y)]
for _ in range(3000):
    w_gd = w_gd - eta * bce_grad(w_gd, X, y)
    loss_gd.append(bce(w_gd, X, y))

# 2. 牛顿法 / IRLS
w_nt = np.zeros(3)
loss_nt = [bce(w_nt, X, y)]
for k in range(8):
    p = sigmoid(X @ w_nt)
    s = p * (1 - p)
    z_work = X @ w_nt + (y - p) / s                      # 工作响应
    w_nt = np.linalg.solve(X.T @ (X * s[:, None]), X.T @ (s * z_work))   # 加权最小二乘
    loss_nt.append(bce(w_nt, X, y))
    print(f'IRLS 第 {k + 1} 步: |grad| = {np.linalg.norm(bce_grad(w_nt, X, y)):.3e}')

# 牛顿步的另一种写法 w - H^{-1} grad, 应该完全一样
w_check = np.zeros(3)
w_check = w_check - np.linalg.solve(bce_hessian(w_check, X), bce_grad(w_check, X, y))
p0 = sigmoid(X @ np.zeros(3)); s0 = p0 * (1 - p0)
w_irls1 = np.linalg.solve(X.T @ (X * s0[:, None]), X.T @ (s0 * (X @ np.zeros(3) + (y - p0) / s0)))
print(f'牛顿步 == IRLS 加权最小二乘? {np.allclose(w_check, w_irls1)}')

# 3. sklearn: C 取很大, 相当于去掉正则化
sk = LogisticRegression(C=1e12, tol=1e-12, max_iter=10000).fit(X_raw, y)
w_sk = np.r_[sk.intercept_, sk.coef_.ravel()]
print(f'梯度下降 3000 步: w = {w_gd}')
print(f'牛顿法 8 步:      w = {w_nt}')
print(f'sklearn:          w = {w_sk}')
print(f'牛顿法 == sklearn? {np.allclose(w_nt, w_sk, atol=1e-5)}, 梯度下降 == 牛顿法? {np.allclose(w_gd, w_nt, atol=1e-3)}')
print(f'predict_proba 一致? {np.allclose(sigmoid(X @ w_nt), sk.predict_proba(X_raw)[:, 1], atol=1e-6)}')

plt.figure(figsize=(4, 3))
J_star = loss_nt[-1]
plt.semilogy(np.array(loss_gd) - J_star + 1e-16, label='gradient descent')
plt.semilogy(np.array(loss_nt) - J_star + 1e-16, 'o-', label='Newton / IRLS')
plt.xscale('symlog')
plt.ylim(1e-14, 1)
plt.xlabel('iteration')
plt.ylabel('J(w) - J(w*)')
plot_nicely()
plt.show()

<a id="decision-boundary"></a>

$$
\text{决策边界 (Decision Boundary): 预测概率正好等于 0.5 的点} \\
\sigma(w^T x) = 0.5 \iff w^T x = 0 \iff w_0 + w_1 x_1 + w_2 x_2 = 0 \implies x_2 = -\frac{w_0 + w_1 x_1}{w_2} \\
\text{逻辑回归的决策边界是一条直线 (高维时是超平面), 所以它是线性分类器} \\
\text{概率的等高线 } \sigma(w^T x) = c \iff w^T x = \text{logit}(c) \text{ 都是和边界平行的直线} \\
\text{(想要曲线边界: 加入 } x_1^2, x_1 x_2 \text{ 等特征, 或者用神经网络)}
$$

In [ ]:
pred = (sigmoid(X @ w_nt) >= 0.5).astype(int)
print(f'训练集准确率: 手算 = {np.mean(pred == y):.4f}, sklearn = {sk.score(X_raw, y):.4f}')

g1, g2 = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
prob = sigmoid(w_nt[0] + w_nt[1] * g1 + w_nt[2] * g2)
x_line = np.linspace(-4, 4, 50)

plt.figure(figsize=(4, 3))
cs = plt.contourf(g1, g2, prob, levels=np.linspace(0, 1, 11), cmap='RdBu_r', alpha=0.4)
plt.colorbar(cs, label='P(y=1|x)')
plt.scatter(X0[:, 0], X0[:, 1], s=8, c='b', label='y = 0')
plt.scatter(X1[:, 0], X1[:, 1], s=8, c='r', label='y = 1')
plt.plot(x_line, -(w_nt[0] + w_nt[1] * x_line) / w_nt[2], 'k', label='w^T x = 0')
plt.xlim(-4, 4)
plt.ylim(-4, 4)
plot_nicely()
plt.show()